# Project 1: $k$-Anonymity
James Bui, Bach Nguyen

This project explore application of the k-Anonymity on the Adult Census Income dataset.

In [1]:
import os
import os.path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Loading in the data

In [ ]:
datadir = "data"
data = os.path.join(datadir, "adult.data")
df = pd.read_csv(data)
df.columns = ["age", "workclass", "fnlwgt", "education", "education-num", 
              "marital-status", "occupation", "relationship", "race", "sex", 
              "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]

# Dropping education-num
df.drop(columns=["education-num"], inplace=True)

## Utility definition

We use three different metrics as defined below:

In [261]:
def avg_equiv_class_size_metric(data, qids, k):
    """
    Return the average sizes of the equivalence classes with respect to the QID set.
    Input:
        data: The input k-anonymized dataframe
        qids: the set of QIDs
    """
    eq_classes = data.groupby(list(qids)).size()
    return eq_classes.mean() / k


def discernability_metrics(data, qids):
    """
    Return the discernability score of the dataset with respect to the QID set.
    The discernability is calculated by assigning the penalty to each tuple depending 
    on the how many tuples are indistinguishable from it.
    Input:
        data (pandas.DataFrame): The input k-anonymized dataframe
        qids (set): the set of QIDs
    """
    eq_classes = data.groupby(list(qids)).size()
    return (eq_classes ** 2).sum()


def classification_metrics(data, qids, sensitive_attr):
    """
    Return the classification metric score of the data with respect to the QID set,
    where we assign a penalty to each tuple t. If t's sensitive attribute matches 
    the majority sensitive attribute, the penalty = 0. Otherwise, penalty = size of the equivalence class.
    Input:
        data (pandas.DataFrame): The input k-anonymized dataframe
        qids (set): the set of QIDs
    """
    penalties = 0
    for _, group in data.groupby(list(qids)):
        class_size = len(group)
        majority = group[sensitive_attr].value_counts().idxmax()
        mismatches = group[group[sensitive_attr] != majority]
        penalties += class_size * len(mismatches)
    return penalties / len(data)

## Generalization of the different quasi-identifiers

In [262]:
def generalize_age(age, level=2, min_age=19, max_age=87):
    """
    Perform dynamic age generalization based on dataset min/max values.

    Input:
        age (int): The original age value to generalize
        level (int): The generalization level
            1 = raw (no generalization)
            2 = dynamic 5-year bins (based on dataset range)
            3 = 10-year bins (optional)
            4 = 20-year bins (optional)
            5 = broad groups (e.g., child, adult, senior)
        min_age (int): Minimum age in the dataset (used for dynamic binning)
        max_age (int): Maximum age in the dataset (used for dynamic binning)

    Output:
        int or str: The generalized age value, depending on the selected level
    """
    if level == 1:
        return age  # raw

    elif level == 2:

        # Calculate which 5-year bin this age falls into
        start = min_age + 5 * ((age - min_age) // 5)
        end = start + 4
        if end > max_age:
            end = max_age
        return f"{start}-{end}"

    elif level == 3:  # 10-year bins
        start = min_age + 10 * ((age - min_age) // 10)
        end = start + 9
        if end > max_age:
            end = max_age
        return f"{start}-{end}"

    elif level == 4:  # 20-year bins
        start = min_age + 20 * ((age - min_age) // 20)
        end = start + 19
        if end > max_age:
            end = max_age
        return f"{start}-{end}"

    elif level == 5:  # broad groups
        if age <= 24: return "<=24"
        elif age <= 44: return "25-44"
        elif age <= 64: return "45-64"
        else: return ">=65"
    elif level > 5:
        return "Any"

    return age

def generalize_workclass(wc, level=3):
    """
    Perform workclass generalization into broader categories.

    Input:
        wc (str): The original workclass value
        level (int): The generalization level
            1 = raw (no generalization)
            2 = Government, Self-employed, Private, Other/None
            3 = Government, Self-employed, Other
            4 = Working, Not working
            5+ = Any

    Output:
        str: The generalized workclass value, depending on the selected level
    """
    wc = wc.strip()
    if level == 2:
        if wc in ["Federal-gov", "Local-gov", "State-gov"]:
            return "Government"
        elif wc in ["Self-emp-inc", "Self-emp-not-inc"]:
            return "Self-employed"
        elif wc in ["Private"]:
            return "Private"
        elif wc in ["Without-pay", "Never-worked"]:
            return "Other/None"
    elif level == 3:
        if wc in ["Federal-gov", "Local-gov", "State-gov"]:
            return "Government"
        elif wc in ["Self-emp-inc", "Self-emp-not-inc"]:
            return "Self-employed"
        else:
            return "Other"
    elif level == 4:
        if wc in ["Private", "Self-emp-inc", "Self-emp-not-inc", "Federal-gov", "Local-gov", "State-gov"]:
            return "Working"
        else:
            return "Not working"
    elif level >= 5:
        return "Any"
    return wc

def generalize_country(c, level=3):
    """
    Perform country generalization into broader regions or categories.

    Input:
        c (str): The original country value
        level (int): The generalization level
            1 = raw (no generalization)
            2 = group into North America, Europe, Asia, Latin America, Unknown, Other
            3 = US, Non-US, Unknown
            4 = Domestic (US), Foreign/Unknown
            5+ = Any

    Output:
        str: The generalized country value, depending on the selected level
    """
    c = c.strip()
    NA = ['United-States','Canada','Puerto-Rico','Outlying-US(Guam-USVI-etc)',
          'Honduras','Mexico','Cuba','Jamaica','Trinadad&Tobago']
    EU = ['England','Germany','Italy','Poland','France','Yugoslavia','Scotland',
          'Greece','Ireland','Hungary','Holand-Netherlands']
    AS = ['India','Iran','Philippines','Cambodia','Thailand','Laos','Taiwan',
          'China','Japan','Vietnam','Hong']
    LATAM = ['Columbia','Ecuador','Haiti','Dominican-Republic','El-Salvador',
             'Guatemala','Nicaragua','Peru','South']

    if level == 2:
        if c in NA: return "North America"
        if c in EU: return "Europe"
        if c in AS: return "Asia"
        if c in LATAM: return "Latin America"
        if c == '?': return "Unknown"
        return "Other"
    elif level == 3:
        if c == "United-States": return "US"
        if c == "?": return "Unknown"
        return "Non-US"
    elif level == 4:
        return "Domestic" if c == "United-States" else "Foreign/Unknown"
    elif level >= 5:
        return "Any"
    return c

def generalize_marital(ms, level=3):
    """
    Perform marital status generalization into broader categories.

    Input:
        ms (str): The original marital status value
        level (int): The generalization level
            1 = raw (no generalization)
            2 = Married, Previously married, Never married
            3 = With partner, Without partner
            4+ = Any

    Output:
        str: The generalized marital status value, depending on the selected level
    """
    ms = ms.strip()
    if level == 1:
        return ms
    elif level == 2:
        if ms in ["Married-civ-spouse", "Married-AF-spouse", "Married-spouse-absent"]:
            return "Married"
        elif ms in ["Divorced", "Separated", "Widowed"]:
            return "Previously married"
        elif ms == "Never-married":
            return "Never married"
    elif level == 3:
        if ms in ["Married-civ-spouse", "Married-AF-spouse", "Married-spouse-absent"]:
            return "With partner"
        else:
            return "Without partner"
    elif level >= 4:
        return "Any"
    return ms

def generalize_education(ed, level=3):
    """
    Perform education generalization into broader categories.

    Input:
        ed (str): The original education level value
        level (int): The generalization level
            1 = raw (no generalization)
            2 = Primary, Secondary, Postsecondary, Tertiary
            3 = Low, Medium, High
            4 = Low, High
            5+ = Any

    Output:
        str: The generalized education value, depending on the selected level
    """
    ed = ed.strip()
    primary = ['Preschool','1st-4th','5th-6th','7th-8th']
    secondary = ['9th','10th','11th','12th','HS-grad']
    postsecondary = ['Some-college','Assoc-acdm','Assoc-voc']
    tertiary = ['Bachelors','Masters','Doctorate','Prof-school']

    if level == 1:
        return ed  # raw

    elif level == 2:  # 4 groups
        if ed in primary: return 'Primary'
        if ed in secondary: return 'Secondary'
        if ed in postsecondary: return 'Postsecondary'
        if ed in tertiary: return 'Tertiary'

    elif level == 3:  # Low / Medium / High
        if ed in primary + secondary: return 'Low'
        if ed in postsecondary: return 'Medium'
        if ed in tertiary: return 'High'

    elif level == 4:  # Low vs High
        if ed in tertiary: return 'High'
        return 'Low'

    elif level >= 5:  # Any
        return "Any"

    return ed

def generalize_race(r, level=3):
    """
    Perform race generalization into broader categories.

    Input:
        r (str): The original race value
        level (int): The generalization level
            1 = raw (no generalization)
            2 = White, Black, Asian, Other
            3 = Majority, Minority
            4+ = Any

    Output:
        str: The generalized race value, depending on the selected level
    """
    r = r.strip()
    if level == 2:
        if r == "White": return "White"
        if r == "Black": return "Black"
        if r == "Asian-Pac-Islander": return "Asian"
        return "Other"
    elif level == 3:
        if r == "White": return "Majority"
        return "Minority"
    elif level >= 4:
        return "Any"
    return r


Applying the generalization

In [263]:
def apply_generalization(df, levels):
    """
    Apply attribute generalization to the dataset according to the specified levels.

    Input:
        df (pandas.DataFrame): The input dataframe containing quasi-identifiers
        levels (dict): A mapping of attribute names to generalization levels
    Output:
        pandas.DataFrame: A copy of the dataframe with generalized attributes
    """
    df = df.copy()
    min_age = df['age'].min()
    max_age = df['age'].max()

    df['age'] = df['age'].apply(lambda x: generalize_age(x, level=levels.get("age", 1), min_age=min_age, max_age=max_age))
    df["education"] = df["education"].apply(lambda x: generalize_education(x, levels.get("education", 1)))
    df["workclass"] = df["workclass"].apply(lambda x: generalize_workclass(x, levels.get("workclass", 1)))
    df["native-country"] = df["native-country"].apply(lambda x: generalize_country(x, levels.get("country", 1)))
    df["marital-status"] = df["marital-status"].apply(lambda x: generalize_marital(x, levels.get("marital-status", 1)))
    df["race"] = df["race"].apply(lambda x: generalize_race(x, levels.get("race", 1)))
    return df

def run_k_anonymization(data, k, levels, qids, sensitive_attr, output_csv="generalized.csv"):
    """
    Run k-anonymization on the dataset with the specified generalization levels and QIDs.

    Input:
        data (pandas.DataFrame): The input dataframe
        k (int): The minimum equivalence class size required for k-anonymity
        levels (dict): A mapping of attributes to generalization levels
        qids (list): The set of quasi-identifiers
        sensitive_attr (str): The sensitive attribute for utility metric evaluation
        output_csv (str, default="generalized.csv"): Path to save the generalized dataframe

    Output:
        tuple:
            - pandas.DataFrame: The k-anonymized dataframe
            - dict: Utility metrics including:
                {
                    "avg_eq_size": average equivalence class size metric,
                    "discernability": discernability metric,
                    "classification": classification metric
                }
    """
    # Drop fnlwgt if present
    if "fnlwgt" in data.columns:
        data = data.drop(columns=["fnlwgt"])

    # Apply manual generalization
    gen_data = apply_generalization(data, levels)

    # 3. Enforce k-anonymity: suppress rare equivalence classes
    eq_sizes = gen_data.groupby(qids).size()
    min_class_size = eq_sizes.min()

    satisfies = min_class_size >= k
    print("K-anonymity:", satisfies)

    # Save to CSV
    gen_data.to_csv(output_csv, index=False)

    # Compute utility metrics
    avg_eq = avg_equiv_class_size_metric(gen_data, qids, k)
    discern = discernability_metrics(gen_data, qids)
    classif = classification_metrics(gen_data, qids, sensitive_attr)

    return gen_data, {
        "avg_eq_size": avg_eq,
        "discernability": discern,
        "classification": classif
    }

In [264]:
def compare_k_levels(data, k_values, levels, qids, sensitive_attr):
    """
    Run k-anonymization for multiple k values and compare utility metrics.

    Input:
        data (pandas.DataFrame): The input dataframe
        k_values (list): A list of k values to evaluate (e.g., [2, 5, 10])
        levels (dict): A mapping of attributes to generalization levels
        qids (list): The set of quasi-identifiers
        sensitive_attr (str): The sensitive attribute for utility metric evaluation

    Output:
        pandas.DataFrame: A dataframe containing utility metrics for each k value
    """
    results = []
    
    for k in k_values:
        gen_data, metrics = run_k_anonymization(
            data, k, levels, qids, sensitive_attr,
            output_csv=f"generalized_k{k}.csv"
        )
        metrics["k"] = k
        results.append(metrics)

    # Convert to DataFrame
    results_df = pd.DataFrame(results)

    return results_df


## Testing and Results

Now we run the $k$-anonymization

In [265]:
# Different k-values to test
k_tests = [2, 4, 6, 15]

levels_list = [
    {
        "age": 4,
        "education": 3,
        "race": 5,
        "country": 5,
        "marital-status": 5,
        "workclass": 2,
    },
    {
        "age": 4,
        "education": 4,
        "race": 5,
        "country": 5,
        "marital-status": 5,
        "workclass": 3,
    },
    {
        "age": 4,
        "education": 5,
        "race": 5,
        "country": 5,
        "marital-status": 5,
        "workclass": 4,
    },
    {
        "age": 4,
        "education": 5,
        "race": 5,
        "country": 5,
        "marital-status": 5,
        "workclass": 4,
    },
]

# QIDs and sensitive attribute
QIDs = ["age", "workclass", "education", "race", "native-country", "marital-status"]
sensitive_attr = "income"

# Run tests in a loop
for k_val, levels in zip(k_tests, levels_list):
    results_df = compare_k_levels(df, k_values=[k_val], levels=levels, qids=QIDs, sensitive_attr=sensitive_attr)
    print(f"Results for k={k_val}:")
    print(results_df)
    print("-" * 50)


K-anonymity: True
Results for k=2:
   avg_eq_size  discernability  classification  k
0   307.169811        89334290      422.146744  2
--------------------------------------------------
K-anonymity: True
Results for k=4:
   avg_eq_size  discernability  classification  k
0   339.166667       182497262      775.304515  4
--------------------------------------------------
K-anonymity: True
Results for k=6:
   avg_eq_size  discernability  classification  k
0   678.333333       392534352     2839.001873  6
--------------------------------------------------
K-anonymity: True
Results for k=15:
   avg_eq_size  discernability  classification   k
0   271.333333       392534352     2839.001873  15
--------------------------------------------------
